## Overview


Apache spark is an open source analytics engine for large scale data processing.

Features:
- Native support for:
    - SQL
    - Streaming data
    - Maching learning
- Spark performance is delivered through *in-memory* computation which accounts for its speed
- Apache Spark is Open Source and provides a consistent interface across platforms
- Integrates with all major cloud vendors
- DataBricks was founded by the creators of Spark and contributes to the open source development effort still

## Spark Components

![](/Volumes/workspace/pyspark_learning/raw_files/images/spark_components.png)

- Spark Core Engine provides the foundation of the system.  It handles memory mangement, fault tolerance, scheduling, and task distribution
- Spark Core Engine is interacted with via high-level APIs, DataFrame, RDD API, and SQL API
- There are specialized API's which 'sit on top' of the DataFrame API, see image above.

The RDD API is the original, low-level Application Programming Interface in Apache Spark.  **RDD stands for Resilient Distributed Dataset**

## Architecture

![](/Volumes/workspace/pyspark_learning/raw_files/images/spark_arch.png)

- **Spark Driver** contains the SparkSession/Context, is the 'brain' of the process, issuing instructions
- **Cluster Manager** manages Spark resources, assigns tasks to the Workers
- **Workers** are cluster nodes which host the executors
- **Executors** process the **tasks** issued by the driver

Driver --> Cluster Mananger --> Worker(s) --> Executor --> Task

At a high level
- Driver assigns tasks to the Workers which host Executors witch run the tasks.
- The Cluster Manager manages the resources themselves, The Workers



### 1. Spark Driver

This is the 'Brain' of the spark application and is what the end-user interacts with

The primary function of the Spark Driver is:
The Spark Driver analyzes the application and creates a Directed Acyclic Graph (DAG) which is a computer science modeling component.

Further:

The Spark Driver creates the SparkSession and is the entry point for all applications.  When in the DBX UI and working in a notebook, the SparkSession has been created for you behind the scenes as '**spark**'.  If working in VSCode or other, one needs to instanciate their own SparkSession manually. In Databricks a default instance of the SparkSession can be used by invoking its methods.  Example Command:


```
spark.createDataFrame(data, schema=schema1)
```

Note: You can manually create additional spark sessions in addition to the default which databricks provides

While providing the unified entry point, the technical function of the driver is:

To analyze the application, construct the Directed Acyclic Graph (DAG), schedule tasks, and monitor execution progress.
Returning results to the client.


### Spark Session

In [0]:
"""
creating my own spark session which I DONT need to do in DBX
Need to import the SparkSession class from the pyspark.sql module
'spark' below is my SparkSession
"""
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# the following is supported by standard compute only
# application_id = spark.sparkContext.applicationId
# print(application_id)

In [0]:
""""
create multiple sessions
Below I manually create the 'default' spark session 'spark'
then, I create many more, naming them whatever I want
"""
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# use the spark session to create a dataframe by invoking its methods
spark.createDataFrame(data, schema=schem a1)


# create more sessions
spark_1 = SparkSession.builder.getOrCreate()

# can also create sessions using the following
spark_app = SparkSession.builder.appName("spark_app").getOrCreate()

# can also use the newSession() method call, based on a current session
spark_new = spark_1.newSession()

# all should have the same context NOTE:  sparkContext is not supported in serverless compute
# print(spark.sparkContext == spark_1.sparkContext)

**Spark DAG**

NOTE:  The acyclic structure ensures no infinite loops are entered

![](/Volumes/workspace/pyspark_learning/raw_files/images/dag.png)



**Relationship between logical plan and DAG**

In Spark, Logical Plans directly map to individual tasks with the DAG without any optimization

**Spark Application Execution**

Jobs --> Stages --> Tasks

The application spawns 'jobs' which are broken into 'stages' which are then broken into 'tasks'.  W**here possible Jobs, stages and tasks are run in parallel**, which implies no dependencies between them

### 2. Cluster Manager

From the driver, execution moves to the **Cluster Manager** .   This allocates resources to the application

### 3. Workers
From the Cluser Manager, comes the **Worker**.   The Worker represents a Node in a cluster which hosts an Executor.  The Worker nodes return results to the driver.

### 4. Executor
From the Workers, comes the **Executor**. This is a process on a Worker node which executes the Task assigned by the Driver.  Executors handle task execution and data caching

Task executon and I/O

- Each Worker may host multiple Executors depending on:
  - Available CPU Cores (`spark.executor.cores`)
  - Available Memory (`spark.executor.memory`)


### Application Execution (on a Woker Node)

Below is the application to task hierarchy of application execution

- Jobs are high-level operations triggered by the Spark Application, triggered by actions such as `save()` or `collect()`
- Stages divide jobs into smaller idenpendent units which can run in parallel
- Tasks are the individual units of work within the Stages

All of this is ocurring on the Workers assigned by the cluster manager

![](/Volumes/workspace/pyspark_learning/raw_files/images/applicationExecution.png)

## Cluster Types in Databricks

- **All Purpose Clusters**:   these are interactive clusters which support notebooks, jobs, and dashboards.  They are configurable with auto-termination

- **Job Clusters**:   these are ephemeral (short lived) clusters which start when a job runs and terminate automatically when a job finishes.  They are optimized for non-interactive workloads

- **SQL Warehouses**:  Optimized clusters for SQL query performance.   Provides instant startup and auto-scaling

## Distributed Computing in DBX

#### Key Characteristics

- **Independence**: Each node works independently of other nodes and manages own CPU and memory
- **Scalability**: Addition of nodes provides more processing power
- **Fault Tolerance**:  Failures are isolated
- **Resource Partitioning**:  Data and workloads are partitioned across nodes

## Dataframes

Dataframes are distributed collections of records, all with the same schema

Dataframes are evaluated as DAGs using 'lazy-evaluation'

**Tungsten** is Sparks columnar in-memory execution engine for consistent representation of dataframes regardless of source data.   Tungsten provides statistics when saving a dataframe to optimized formats (parquet and delta)

## Plans

Spark optimizes the execution plan through several stages (Unresolved Logical Plan to Physical Plan).


**Catalyst Optimizer**:   This engine applies _rule-based and cost-based optimizations_ to convert the DataFrame operations into an optimized execution plan, working in conjunction with the vectorized Photon Engine

**Photon Engine**:  Optimizes query execution processing data in batches rather than row-by-row.  Runs by default in SQL warehouse and serverless compute.  Can be enabled on all-purpose compute and job clusters

Serverless compute - Photon is always on

Classic compute - Photon can be disabled/enabled

Dataframes are immuatable.  Once created they cannot be modified.   Transformations create new dataframs from existing dataframes.

## whats my schema?

```dataframe_name.printShema()```:  this is a method on the schema which will list the columns and datatypes

```print(dataframe_name.schema)```:   this is the pythonic way to display the schema.  The datatypes will be displayed as python StructFields and StructTypes

Inferring a schema has higher overhead as it requires spark jobs to launch.  It has to analyze the data then create the dataframe.

Supplying a schema has lower overhead as no spark jobs are created to process the data, its just a simple transformation

DDL schemas are best for flat data

Python schemas are best for complex, nested data

## Lazy Evaluation

- Lazy evaluation in Spark means that transformations on DataFrames (like filter, select, map) are not executed immediately.
- Instead, Spark builds a logical plan and only executes the computation when an action (like ```display(), count(), collect()```) is called.
- This allows Spark to optimize the execution plan before running it.

In [0]:
# Example:
df = spark.read.format("csv").option("header", "true").load("/databricks-datasets/nyctaxi/taxidata.csv")
filtered_df = df.filter(df["passenger_count"] > 2)  # Transformation (lazy)


## Actions vs Transformations

- Spark transformations (e.g., filter, select, map) create a new DataFrame from an existing one and are lazy, they do not trigger computation.
- Spark actions (e.g., display(), count(), collect()) trigger the execution of the transformations and return results or output.

In [0]:
# Example transformation (lazy, does not execute immediately):
transformed_df = df.select("passenger_count").filter(df["passenger_count"] > 2)

# Example action (triggers execution):
display(transformed_df)

### UDFs

UDFs can extend the capabilities of Databricks; however:
- UDFs cannot be optimized by the Catalyst Optimizer and require additional serialization overhead
- Always use a built-in function when possible